# Drifting Diffusing Gaussian

This is a simple case to develop my own framework on top of FEniCSx. It solves the depth-averaged advection-diffusion equation on a square grid.

$$
\begin{cases}
    \frac{\mathrm{d} \iota}{\mathrm{d} t} + \nabla \cdot (\iota \mathbf{v} - \mathbf{D} \nabla c) = 0 &x \in \Omega \\
    \iota = \iota_{in} &x \in \Gamma_{\mathrm{in}} \\
    \mathbf{D} \nabla c \cdot \mathbf{\hat{n}} = 0 & x \in \Gamma_{\mathrm{out}}
\end{cases}
$$

## Imports and Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import dolfinx as dx
import dolfinx.fem.petsc
import dolfinx.plot as dxp
import fenicsxtools as ft
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import ufl
import ufl.formatting.ufl2unicode

In [3]:
pv.set_jupyter_backend('static')
plotter = pv.Plotter()

## Problem Parameters

Here, we give more specifics for the problem. Specifically we are solving for a constant rightward flow and uniform diffusion in a square domain. The initial condition is a Gaussian bump, which should simply move rightwards and spread.

$$
\begin{align}
\mathbf{v} &= v_{0} \mathbf{\hat{e}}_{x} \\
\mathbf{D} &= D \mathbf{I} \\
\iota_{0}(\mathbf{x}) &= e^{-\frac{|\mathbf{x} - \mathbf{x}_{0}|^{2}}{2 \sigma^{2}}} \\
\iota(\mathbf{x}, t) &= \frac{\sigma^{2}}{\sigma^{2} + 2 d t} e^{-\frac{|\mathbf{x} - \mathbf{x}_{0} - \mathbf{v} t|^{2}}{2 (\sigma^{2} + 2 D t)}}
\end{align}
$$

In [4]:
# Equation parameters
h0 = 1.0 # Constant height
u0 = 1.0 # Constant rightward flow
v0 = 0.0 # No vertical flow
d0 = 0.05 # Isotropic diffusion

In [5]:
# Initial condition
x0 = 0.5
y0 = 0.5
r0 = 0.1
def iota0(x):
    return np.exp(-((x[0] - x0) ** 2 + (x[1] - y0) ** 2)/ (2 * r0 ** 2))

In [6]:
# True solution
def iota_true_function(x, t):
    return r0 ** 2 / (r0 ** 2 + 2 * d0 * t) * np.exp(-((x[0] - x0 - u0 * t) ** 2 + (x[1] - y0 - v0 * t) ** 2)/ (2 * (r0 ** 2 + 2 * d0 * t)))

## Discretization and Meshing Parameters

A simple unit square $\Omega = (0, 1)^{2}$ is used for the domain.

In [7]:
# Numerical parameters
nx = 10 # Cell resolution
dt = 0.001 # Time step
nt = 1000
ntps = 1 / dt
write_every = 10 # Only every nth frame
plot_speedup = 1 # Relative to sim time

In [8]:
# Define a simple
domain = dx.mesh.create_unit_square(MPI.COMM_WORLD, nx, nx, cell_type=dx.mesh.CellType.triangle)

## FEM Formulation

Here my library comes into use.
- The user defines function spaces and relevant functions.
- A template for depth-averaged advection-diffusion assembles them into the correct PDE.
- This is then fed through a formulation, which constructs the appropriate semidiscrete equations.
- The semidiscrete equations are then solved, either implicitly or explicitly.

In [9]:
# Declare function spaces
# For now I'm using piecewise linear basis functions
V = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1))
W = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1, (domain.geometry.dim,)))

In [10]:
# Initialize functions (solution variables and field variables)
iota = dx.fem.Function(V) # The conserved quantity
iota.interpolate(iota0)
# diota_dt = dx.fem.Function(V) # The derivative
h = dx.fem.Constant(domain, h0) # Uniform water column height
v = dx.fem.Constant(domain, (u0, v0)) # Uniform velocity
D = d0 * ufl.Identity(domain.geometry.dim) # Uniform isotropic diffusion

In [11]:
# Generate strong form
# A depth-averaged advection-diffusion template is used
equation = ft.equations.get_depth_averaged_advection_diffusion(
    domain=domain,
    iota=iota,
    h=h,
    v=v,
    D=D
)

In [12]:
# Check equation
# dU/dt + div(-1 * 0.05 * I * grad(w / c0) + c1 w) = 0
# J = c1
print(equation)
print(ufl.formatting.ufl2unicode.ufl2unicode(ufl.algorithms.ad.expand_derivatives(equation.J)))

dU/dt + div([-1 [∑[i₁₃]([0.05 I[i₈,i₉] ∀ i₈,i₉][i₁₂,i₁₃] [(𝐠𝐫𝐚𝐝 w₀)[i₁₅] ∕ c⁰ ∀ i₁₅][i₁₃]) ∀ i₁₂][i₁₁] ∀ i₁₁] + [c⃗¹[i₁₀] w₀ ∀ i₁₀]) = 0
c⃗¹


In [13]:
# Choose formulation to build weak forms
# An alternating or downwind flux is generally used for the gradient
formulation = ft.formulations.LDGFormulation(
    equation=equation,
    grad_of=iota / h,
    space_auxiliary=W,
    trace_function_q=ft.fluxes.fluxn_upwind_scalar,
    trace_function_r=ft.fluxes.trace_downwind
)

## To be abstracted...

- TS object for semidiscrete equations

In [14]:
# Get form for q mass matrix
M_q_form = dx.fem.form(formulation.mass_form_q)

In [15]:
# Assemble mass matrix
M_q = dx.fem.petsc.assemble_matrix(M_q_form)
M_q.assemble()

In [16]:
# Set up solver
# Serial only
M_q_solver = PETSc.KSP().create(domain.comm)
M_q_solver.setOperators(M_q)
M_q_solver.setType(PETSc.KSP.Type.PREONLY)
M_q_solver.getPC().setType(PETSc.PC.Type.LU)
M_q_solver.setFromOptions() # Allows command-line PETSc options to override the above
M_q_solver.setUp()

In [17]:
# Get form for q residual
g_q_form = dx.fem.form(formulation.residual_form_q)

In [18]:
# Assemble residual
G_q_vec = dx.fem.petsc.create_vector(V)
b_q = dx.fem.petsc.create_vector(V)

In [19]:
# Get form for r mass matrix
M_r_form = dx.fem.form(formulation.mass_form_r)

In [20]:
# Assemble mass matrix
M_r = dx.fem.petsc.assemble_matrix(M_r_form)
M_r.assemble()

In [21]:
# Set up solver
# Serial only
M_r_solver = PETSc.KSP().create(domain.comm)
M_r_solver.setOperators(M_r)
M_r_solver.setType(PETSc.KSP.Type.PREONLY)
M_r_solver.getPC().setType(PETSc.PC.Type.LU)
M_r_solver.setFromOptions() # Allows command-line PETSc options to override the above
M_r_solver.setUp()

In [22]:
# Get form for r residual
g_r_form = dx.fem.form(formulation.residual_form_r)

In [23]:
# Assemble residual
G_r_vec = dx.fem.petsc.create_vector(W)
b_r = dx.fem.petsc.create_vector(W)

In [24]:
# Set up left hand side
# Not needed

In [25]:
# Set up approximate Jacobian
# Not needed

In [26]:
# Store auxiliary variable
g_vec = dx.la.petsc.create_vector_wrap(formulation.r.x)

In [27]:
# Set up right hand side
def rhs(ts, t, u, G):
    """Right-hand side G of the general TS ODE.

    F(t, u, du/dt) = G(t, u)

    Arguments:
        ts: A PETSc time stepper object.
        t: The current time.
        u: The PETSc state vector.
        G: The PETSc residual value.

    Returns:
        None: The function sets a new value for G.
    """
    # In this case, G = M^-1 R(u)
    # So there is both an update of the residual and a matrix solve

    # Update my function iota from TS function x
    dx.fem.petsc.assign(u, iota)
    iota.x.scatter_forward()
    
    # Residual of r update
    with b_r.localForm() as b_r_local:
        b_r_local.set(0.0) # Flush everything to 0
    dx.fem.petsc.assemble_vector(b_r, g_r_form)
    b_r.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)

    # Matrix solve
    # No need to update M from the weak form
    # Result goes to g_vec
    M_r_solver.solve(b_r, g_vec)
    g_vec.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)

    # Residual of q update
    with b_q.localForm() as b_q_local:
        b_q_local.set(0.0) # Flush everything to 0
    dx.fem.petsc.assemble_vector(b_q, g_q_form) # Update using weak form
    b_q.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    
    # Matrix solve
    # No need to update M from weak form
    # Result goes to g, the "return value"
    M_q_solver.solve(b_q, G)
    G.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)

In [28]:
# Create PETSc TS solver
ts = PETSc.TS().create(domain.comm)

In [29]:
# Set time stepping scheme
ts.setType(PETSc.TS.Type.EULER)
ts.setTime(0.0)
ts.setTimeStep(dt)
ts.setMaxTime(dt * nt)
ts.setExactFinalTime(PETSc.TS.ExactFinalTime.MATCHSTEP)

In [30]:
# Set equation
ts.setRHSFunction(rhs, G_q_vec)
ts.setSolution(iota.x.petsc_vec)

In [31]:
# Set up plotting preliminaries
topology, cell_types, geometry = dxp.vtk_mesh(V)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)

In [32]:
# Numerical plotter
grid.point_data['iota_numerical'] = np.asarray(iota.x.array.real).copy()
plotter_numerical = pv.Plotter()
plotter_numerical.add_mesh(
    grid,
    scalars='iota_numerical',
    cmap='jet',
    clim=[0.0, 1.0],
    show_edges=False
)
plotter_numerical.view_xy()
plotter_numerical.open_gif('drifting_diffusing_gaussian_numerical.gif', fps=(plot_speedup * ntps / write_every))

In [33]:
# True plotter
iota_true = dx.fem.Function(V)
iota_true.interpolate(lambda x : iota_true_function(x, 0.0))
grid.point_data['iota_true'] = np.asarray(iota_true.x.array.real).copy()
plotter_true = pv.Plotter()
plotter_true.add_mesh(
    grid,
    scalars='iota_true',
    cmap='jet',
    clim=[0.0, 1.0],
    show_edges=False
)
plotter_true.view_xy()
plotter_true.open_gif('drifting_diffusing_gaussian_true.gif', fps=(plot_speedup * ntps / write_every))

In [34]:
# Error plotter
grid.point_data['iota_error'] = np.asarray(iota.x.array.real - iota_true.x.array.real).copy()
plotter_error = pv.Plotter()
plotter_error.add_mesh(
    grid,
    scalars='iota_error',
    cmap='RdBu',
    clim=[-0.1, 0.1],
    show_edges=False
)
plotter_error.view_xy()
plotter_error.open_gif('drifting_diffusing_gaussian_error.gif', fps=(plot_speedup * ntps / write_every))

In [35]:
# Set monitor for time stepping loop
def monitor(ts, step, t, u):
    if step % write_every != 0:
        return
    # Make sure solution is updated
    dx.fem.petsc.assign(u, iota)
    iota.x.scatter_forward()
    iota_true.interpolate(lambda x : iota_true_function(x, t))
    # Plotting from rank 0
    if domain.comm.rank == 0:
        grid.point_data['iota_numerical'] = np.asarray(iota.x.array.real).copy()
        grid.point_data['iota_true'] = np.asarray(iota_true.x.array.real).copy()
        grid.point_data['iota_error'] = np.asarray(iota.x.array.real - iota_true.x.array.real).copy()
        plotter_numerical.write_frame()
        plotter_true.write_frame()
        plotter_error.write_frame()
ts.setMonitor(monitor)

In [36]:
# Finalize setup
ts.setFromOptions()

## Solution Loop

In [37]:
# Solution loop
ts.solve(iota.x.petsc_vec)

In [38]:
# Close and save file
plotter_numerical.close()
plotter_true.close()
plotter_error.close()